# Bangla tokenizer behaviour under Qwen3-4B

Measures two properties of the tokenizer relied on by Shi, Hauer, Basil, Zhang and Kondrak, *Word Surprisal Correlates with Sentential Contradiction in LLMs* (EACL 2026).

Section 4 of that paper reconstructs word probabilities from subword token probabilities using a constrained beam search of width 10 and depth 10, plus an end-of-word injection that depends on partitioning the vocabulary into beginning-of-word and mid-word tokens. Footnote 2 notes that the partition depends on the tokenizer and the language.

1. **Fertility** — subword tokens per word in Bangla against English, and the fraction of words exceeding depth 10.
2. **Vocabulary coverage** — which vocabulary entries Bangla text actually draws on, and what fraction of them decode to well-formed text.

Tokenizer only. No model weights, no GPU.

In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B"
N_SENTENCES = 300
BEAM_DEPTH = 10  # D in Algorithm 1 of the paper

BANGLA_BLOCK = range(0x0980, 0x0A00)
REPLACEMENT = "\ufffd"  # U+FFFD, emitted when a token is a partial UTF-8 sequence

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

# Recover the word-start marker by diffing a word with and without a leading space.
# Note: the marker is often U+0120 (G with dot above), which is alphanumeric,
# so character-class tests do not identify it.
_probe = tok.tokenize(" hello")[0]
BOW_MARKER = _probe[: -len("hello")] if _probe.endswith("hello") else ""

print("tokenize(' hello') ->", tok.tokenize(" hello"))
print("word-start marker  ->", repr(BOW_MARKER))
assert BOW_MARKER, "marker not recovered; inspect the probe output above"

## Data

Wikipedia prose in both languages. Comparable registers, not parallel text.

In [ ]:
import re
from datasets import load_dataset


def wiki_sentences(config, splitter, n):
    """Stream a Wikipedia dump and collect n moderately sized sentences."""
    ds = load_dataset("wikimedia/wikipedia", config, split="train", streaming=True)
    out = []
    for article in ds:
        for s in re.split(splitter, article["text"]):
            s = s.strip()
            if 40 < len(s) < 400:
                out.append(s)
                if len(out) >= n:
                    return out
    return out


bn_sentences = wiki_sentences("20231101.bn", r"\u0964", N_SENTENCES)
en_sentences = wiki_sentences("20231101.en", r"(?<=\.)\s+", N_SENTENCES)

print(f"Bangla sentences:  {len(bn_sentences)}")
print(f"English sentences: {len(en_sentences)}")

## 1. Fertility

In [ ]:
import statistics


def token_counts(sentences):
    """Subword token count for each whitespace-delimited word."""
    counts = []
    for sentence in sentences:
        for word in sentence.split():
            counts.append(len(tok.encode(" " + word, add_special_tokens=False)))
    return counts


def report(label, sentences):
    counts = token_counts(sentences)
    if not counts:
        print(f"{label}: no data")
        return None
    over = sum(1 for c in counts if c > BEAM_DEPTH) / len(counts)
    print(label)
    print(f"  words analysed      : {len(counts)}")
    print(f"  mean tokens/word    : {statistics.mean(counts):.2f}")
    print(f"  median tokens/word  : {statistics.median(counts)}")
    print(f"  max tokens/word     : {max(counts)}")
    print(f"  words > depth {BEAM_DEPTH}    : {over:.1%}")
    print()
    return statistics.mean(counts)


bn_mean = report("Bangla", bn_sentences)
en_mean = report("English", en_sentences)

if bn_mean and en_mean:
    print(f"Bangla / English fertility ratio: {bn_mean / en_mean:.2f}x")

## 2. Vocabulary coverage

Scanning the vocabulary for entries that *decode* to Bangla script undercounts
badly: a token holding a partial UTF-8 sequence decodes to U+FFFD and contains
no Bangla codepoint, so it is missed even though Bangla text uses it heavily.

The honest measure is to tokenize the Bangla corpus and look at the entries it
actually draws on.

In [ ]:
from collections import Counter

usage = Counter()
for sentence in bn_sentences:
    usage.update(tok.tokenize(sentence))


def decode(token):
    try:
        return tok.convert_tokens_to_string([token])
    except Exception:
        return REPLACEMENT


wellformed = [t for t in usage if REPLACEMENT not in decode(t)]
partial = [t for t in usage if REPLACEMENT in decode(t)]

tokens_total = sum(usage.values())
tokens_partial = sum(usage[t] for t in partial)

print(f"vocabulary size                       : {len(tok.get_vocab())}")
print(f"distinct entries used by Bangla text  : {len(usage)}")
print(f"  decoding to well-formed text        : {len(wellformed)}")
print(f"  holding a partial UTF-8 sequence    : {len(partial)}")
print()
print(f"token occurrences                     : {tokens_total}")
print(f"  that are partial sequences          : {tokens_partial} ({tokens_partial / tokens_total:.1%})")

In [ ]:
# Standalone dependent signs: orthographically these cannot occur alone,
# so their presence as whole tokens means merges stop inside a grapheme cluster.
DEPENDENT_SIGNS = set(
    "ািীুূৃেৈোৌ্"
    "ঁংঃ"
)

bare_signs = {}
for token in tok.get_vocab():
    text = decode(token).strip()
    if text and all(ch in DEPENDENT_SIGNS for ch in text):
        bare_signs[token] = text

print(f"vocabulary entries that are only dependent signs: {len(bare_signs)}")
for t, s in bare_signs.items():
    print(f"  {t!r:<10} -> {s!r}")

# Which basic Bangla letters have a single-token representation at all.
letters = [chr(c) for c in list(range(0x0985, 0x098D)) + list(range(0x098F, 0x0991))
           + list(range(0x0993, 0x09A9)) + list(range(0x09AA, 0x09B1))]
single = [ch for ch in letters if len(tok.tokenize(ch)) == 1]
multi = [ch for ch in letters if len(tok.tokenize(ch)) > 1]

print(f"\nbasic letters with a single token ({len(single)}): {''.join(single)}")
print(f"basic letters needing several     ({len(multi)}): {''.join(multi)}")

In [ ]:
# Worked example: a single long Bangla word.
EXAMPLE = "\u09ac\u09bf\u09b6\u09cd\u09ac\u09ac\u09bf\u09a6\u09cd\u09af\u09be\u09b2\u09af\u09bc"

pieces = tok.tokenize(" " + EXAMPLE)
print(f"word   : {EXAMPLE}")
print(f"tokens : {len(pieces)}\n")
for p in pieces:
    print(f"  {p!r:<14} -> {decode(p)!r}")